# Entrenamiento ResNet50 - Clasificación de Plantas
### Happy Tree Friends - Transfer Learning con PyTorch

**Pasos antes de correr este notebook:**
1. Comprime la carpeta `data/3 - copia/` a un archivo ZIP llamado `dataset.zip`
2. Súbelo a tu Google Drive (raíz)
3. En Colab: `Runtime → Change runtime type → T4 GPU`
4. Corre las celdas en orden

In [ ]:
# Celda 1: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado correctamente')

In [ ]:
# Celda 2: Descomprimir dataset
import zipfile, os

# Si subiste el zip en una carpeta diferente, ajusta este path
ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
EXTRACT_PATH = '/content/dataset'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_PATH)

print('Carpetas encontradas:', os.listdir(EXTRACT_PATH))

In [ ]:
# Celda 3: Imports y configuración
import torch
import torchvision
import json
import numpy as np
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader, Subset

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR   = '/content/dataset/3 - copia'
BATCH_SIZE = 32
IMG_SIZE   = 224
NUM_EPOCHS_HEAD     = 15   # fase 1: entrenar solo la cabeza
NUM_EPOCHS_FINETUNE = 10   # fase 2: afinar layer4 + cabeza
SEED = 42

torch.manual_seed(SEED)
print(f'Dispositivo: {DEVICE}')
print(f'Versión PyTorch: {torch.__version__}')

In [ ]:
# Celda 4: Transforms y DataLoaders
# IMPORTANTE: usamos dos ImageFolder separados con distintos transforms
# para evitar que train y val compartan la misma augmentación

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Cargar dataset dos veces con distintos transforms
train_dataset = datasets.ImageFolder(DATA_DIR, transform=train_tf)
val_dataset   = datasets.ImageFolder(DATA_DIR, transform=val_tf)

class_names = train_dataset.classes
n_total = len(train_dataset)
n_val   = int(n_total * 0.2)
n_train = n_total - n_val

# Mismos índices para ambos splits (seed fija)
indices = torch.randperm(n_total, generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices = indices[n_val:]
val_indices   = indices[:n_val]

train_ds = Subset(train_dataset, train_indices)
val_ds   = Subset(val_dataset,   val_indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Clases ({len(class_names)}): {class_names}')
print(f'Train: {n_train} imágenes | Val: {n_val} imágenes')

In [ ]:
# Celda 5: Modelo ResNet50 con transfer learning
model = models.resnet50(weights='IMAGENET1K_V1')

# Congelar todas las capas
for param in model.parameters():
    param.requires_grad = False

# Reemplazar la capa final para 7 clases
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 7)
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros totales:     {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}')

In [ ]:
# Celda 6: Función de entrenamiento
def train_model(model, train_loader, val_loader, epochs, optimizer, scheduler=None):
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0

    for epoch in range(epochs):
        # --- Entrenamiento ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
        if scheduler:
            scheduler.step()

        # --- Validación ---
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                _, preds = model(imgs).max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total   += labels.size(0)

        train_acc = 100 * correct / total
        val_acc   = 100 * val_correct / val_total
        best_val_acc = max(best_val_acc, val_acc)

        print(f'Epoch {epoch+1:2d}/{epochs} | '
              f'Loss: {running_loss/len(train_loader):.3f} | '
              f'Train: {train_acc:.1f}% | '
              f'Val: {val_acc:.1f}%')

    print(f'\nMejor val accuracy: {best_val_acc:.1f}%')

In [ ]:
# Celda 7: Fase 1 - Entrenar solo la cabeza clasificadora
print('=== FASE 1: Entrenando solo la cabeza (fc) ===')
optimizer1 = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler1 = optim.lr_scheduler.StepLR(optimizer1, step_size=5, gamma=0.5)
train_model(model, train_loader, val_loader, NUM_EPOCHS_HEAD, optimizer1, scheduler1)

In [ ]:
# Celda 8: Fase 2 - Fine-tuning de layer4 + cabeza
print('=== FASE 2: Fine-tuning layer4 + fc ===')
for param in model.layer4.parameters():
    param.requires_grad = True

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables ahora: {trainable_now:,}')

optimizer2 = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-4},
])
train_model(model, train_loader, val_loader, NUM_EPOCHS_FINETUNE, optimizer2)

In [ ]:
# Celda 9: Guardar modelo y nombres de clases en Google Drive
import json

MODEL_PATH  = '/content/drive/MyDrive/resnet50_plantas.pt'
CLASSES_PATH = '/content/drive/MyDrive/class_names.json'

torch.save(model.state_dict(), MODEL_PATH)
with open(CLASSES_PATH, 'w') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print('Archivos guardados en Google Drive:')
print(f'  Modelo:  {MODEL_PATH}')
print(f'  Clases:  {CLASSES_PATH}')
print(f'\nClases en orden: {class_names}')
print('\nDescarga ambos archivos y ponlos en backend/model/')

In [ ]:
# Celda 10 (opcional): Verificar predicción de prueba
from PIL import Image
import random

model.eval()
# Tomar una imagen aleatoria del val set
idx = random.choice(val_indices)
img_path, true_label = val_dataset.samples[idx]
img = Image.open(img_path).convert('RGB')

tensor = val_tf(img).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    output = model(tensor)
    probs  = torch.softmax(output, dim=1)[0]
    pred_idx = probs.argmax().item()

print(f'Imagen: {img_path.split("/")[-1]}')
print(f'Real:      {class_names[true_label]}')
print(f'Predicho:  {class_names[pred_idx]} ({100*probs[pred_idx]:.1f}%)')
print('\nTodas las probabilidades:')
for i, (cls, p) in enumerate(zip(class_names, probs.tolist())):
    bar = '█' * int(p * 30)
    print(f'  {cls:<35} {bar} {100*p:.1f}%')